In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:


def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)
    shifted_logits = logits - np.max(logits)
    exp_scores = np.exp(shifted_logits)
    probabilities = exp_scores / np.sum(exp_scores)
    return probabilities

def cross_entropy(p, q, epsilon=1e-15):
    """
    Categorical Cross-Entropy: H(P, Q) = - sum_i ( p_i * log(q_i) )
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    q_clipped = np.clip(q, epsilon, 1.0 - epsilon)
    return -np.sum(p * np.log(q_clipped))

def kl_divergence(p, q, epsilon=1e-15):
    """
    Kullback-Leibler Divergence: D_KL(P || Q) = sum_i ( p_i * log(p_i / q_i) )
    Where 0 * log(0) is evaluated as 0 in information theory limits.
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p_safe = np.clip(p, epsilon, 1.0)
    q_clipped = np.clip(q, epsilon, 1.0 - epsilon)
    non_zero = p > 0
    return np.sum(p[non_zero] * np.log(p_safe[non_zero] / q_clipped[non_zero]))



In [ ]:
# ==============================================================================
# STEP 3: EMPIRICAL OPTIMIZATION PATH AUDIT
# ==============================================================================

# 1. Define true one-hot distribution (Class 0 is correct)
p_true = np.array([1.0, 0.0, 0.0])

# 2. Simulate shifting logits where w scales from -2 to 5 (confidence grows)
weights = np.linspace(-2, 5, 100)
simulated_logits = [np.array([w, 1.0, 0.5]) for w in weights]

# 3. Compute loss trajectories
ce_losses = []
kl_losses = []

for logit in simulated_logits:
    q_prob = softmax(logit)
    ce_losses.append(cross_entropy(p_true, q_prob))
    kl_losses.append(kl_divergence(p_true, q_prob))

ce_losses = np.array(ce_losses)
kl_losses = np.array(kl_losses)

# 4. Equivalence Line Graph
plt.figure(figsize=(10, 6), dpi=100)

plt.plot(weights, ce_losses, label="Cross-Entropy: $H(P, Q)$", color="#1f77b4", linewidth=2.5)
plt.plot(weights, kl_losses, label="KL Divergence: $D_{KL}(P \parallel Q)$", color="#ff7f0e", linestyle="--", linewidth=2.5)

# Plot the absolute difference between the two metrics
diff = np.abs(ce_losses - kl_losses)
plt.plot(weights, diff, label="Difference: $|H(P, Q) - D_{KL}(P \parallel Q)| = H(P)$", color="#2ca02c", linestyle=":", linewidth=2)

plt.title("Empirical Equivalence: Cross-Entropy vs. KL Divergence Optimization Path", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Logit Weight for Correct Class ($w$)", fontsize=11)
plt.ylabel("Loss Value", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=10, loc="upper right")
plt.tight_layout()
plt.show()

# Print audit metrics
entropy_p = -np.sum(p_true[p_true > 0] * np.log(p_true[p_true > 0]))
print(f"Empirical verification across {len(weights)} points:")
print(f"True Distribution Entropy H(P): {entropy_p:.6f}")
print(f"Max Absolute Divergence Difference: {np.max(diff):.6e}")
print(f"Min Absolute Divergence Difference: {np.min(diff):.6e}")